# Module 12 — Độ tin cậy, độ giá trị & đạo đức nghiên cứu
Notebook RIÊNG cho Module 12 (module cuối cùng, 12/12) — tái lập Cronbach's Alpha, item-total correlation, alpha-if-deleted cho cả 2 thang đo, và liên hệ với EFA (Module 10) như bằng chứng độ giá trị cấu trúc.

Tương ứng SPSS/PSPP: `Analyze > Scale > Reliability Analysis`, lệnh `RELIABILITY`.

In [1]:
# Chạy ô này đầu tiên (Colab: bấm ▶). Không cần cài gì thêm.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """Bộ dữ liệu GIẢ LẬP 'Lớp học 240 học sinh' — CÙNG một seed=2026 dùng
    xuyên suốt các module 03-08, khớp chính xác với lop_hoc_240.csv/.sav
    trong repo (không phải dữ liệu thật, nhưng tái lập được 100%)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Nam", "Nữ"], n)
    method = rng.choice(["Truyền thống", "Dự án"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Dự án")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Dự án") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("Đã tạo df:", df.shape)

Đã tạo df: (240, 14)


## 1. Cronbach's Alpha — thủ công, đối chiếu PSPP
🎯 **Phương pháp này trả lời câu hỏi gì?** Cronbach's Alpha trả lời: các item trong một thang đo có "đồng hành" NHẤT QUÁN với nhau hay không — đo độ TIN CẬY của thang đo, chưa nói gì tới việc thang đo có đo ĐÚNG khái niệm cần đo (độ giá trị, mục 3).

In [2]:
def cronbach_alpha(items_df):
    items_df = items_df.dropna()
    k = items_df.shape[1]
    item_vars = items_df.var(axis=0, ddof=1)
    total_var = items_df.sum(axis=1).var(ddof=1)
    alpha = (k/(k-1)) * (1 - item_vars.sum()/total_var)
    return alpha, len(items_df)

a_interest, n1 = cronbach_alpha(df[["h1","h2","h3"]])
a_anxiety, n2 = cronbach_alpha(df[["a1","a2","a3"]])
print(f"Cronbach's alpha (Hứng thú, h1-h3): {a_interest:.4f} (n={n1})")
print(f"Cronbach's alpha (Lo âu, a1-a3): {a_anxiety:.4f} (n={n2})")

Cronbach's alpha (Hứng thú, h1-h3): 0.7133 (n=237)
Cronbach's alpha (Lo âu, a1-a3): 0.7585 (n=240)


#### 📤 Đầu ra thật
α(Hứng thú)=0,7133, α(Lo âu)=0,7585. ✅ Khớp tuyệt đối với PSPP (`RELIABILITY`): 0,71 và 0,76 (làm tròn 2 chữ số).

## 2. Item-total correlation & Alpha-if-deleted
🎯 **Phương pháp này trả lời câu hỏi gì?** Hai chỉ số này trả lời: TỪNG item riêng lẻ đóng góp thế nào vào độ tin cậy chung — item nào "kéo" alpha xuống, và xoá bớt item có cải thiện được thang đo không?

In [3]:
items = df[["h1","h2","h3"]].dropna()
total = items.sum(axis=1)
print("--- Thang Hứng thú ---")
for col in items.columns:
    rest = total - items[col]
    r = np.corrcoef(items[col], rest)[0,1]
    others = [c for c in items.columns if c != col]
    a_del, _ = cronbach_alpha(items[others])
    print(f"{col}: item-rest r={r:.3f}, alpha-if-deleted={a_del:.4f}")

--- Thang Hứng thú ---
h1: item-rest r=0.570, alpha-if-deleted=0.5752
h2: item-rest r=0.492, alpha-if-deleted=0.6716
h3: item-rest r=0.535, alpha-if-deleted=0.6195


#### 📤 Đầu ra thật
h1: r=0,570, alpha-if-deleted=0,575. h2: r=0,492, alpha-if-deleted=0,672. h3: r=0,535, alpha-if-deleted=0,620. ✅ Khớp PSPP (0,57/0,58; 0,49/0,67; 0,54/0,62). Xoá bất kỳ item nào cũng làm alpha GIẢM so với 0,713 hiện tại → giữ nguyên cả 3 item là lựa chọn tốt nhất.

## 3. Độ giá trị cấu trúc — liên hệ với EFA (Module 10)
🎯 **Phương pháp này trả lời câu hỏi gì?** Đối chiếu kết quả EFA (Module 10) với thiết kế lý thuyết ban đầu trả lời: thang đo có ĐO ĐÚNG khái niệm lý thuyết dự kiến hay không (độ giá trị cấu trúc) — khác hẳn câu hỏi "có nhất quán không" (độ tin cậy, mục 1-2).

In [4]:
from factor_analyzer import FactorAnalyzer
items6 = df[["h1","h2","h3","a1","a2","a3"]].dropna()
fa = FactorAnalyzer(n_factors=2, rotation="varimax")
fa.fit(items6)
loadings = pd.DataFrame(fa.loadings_, index=items6.columns, columns=["Nhân tố 1", "Nhân tố 2"])
print(loadings.round(3))
print("\n-> h1-h3 tải rõ lên 1 nhân tố, a1-a3 tải rõ lên nhân tố còn lại")
print("-> Khớp đúng thiết kế lý thuyết ban đầu (2 khái niệm: hứng thú, lo âu)")
print("-> Đây là bằng chứng CỦNG CỐ độ giá trị cấu trúc của 2 thang đo")

    Nhân tố 1  Nhân tố 2
h1     -0.043      0.747
h2     -0.003      0.603
h3     -0.096      0.670
a1      0.725     -0.092
a2      0.722     -0.083
a3      0.695     -0.016

-> h1-h3 tải rõ lên 1 nhân tố, a1-a3 tải rõ lên nhân tố còn lại
-> Khớp đúng thiết kế lý thuyết ban đầu (2 khái niệm: hứng thú, lo âu)
-> Đây là bằng chứng CỦNG CỐ độ giá trị cấu trúc của 2 thang đo


#### 📤 Đầu ra thật
Cấu trúc EFA khớp hoàn toàn với thiết kế ban đầu (giống Module 10) — kết hợp với Cronbach's Alpha chấp nhận được ở mục 1, ta có bằng chứng khá vững cho cả ĐỘ TIN CẬY (alpha) VÀ ĐỘ GIÁ TRỊ CẤU TRÚC (EFA khớp thiết kế) của 2 thang đo này.

## 4. Minh hoạ "tin cậy cao, giá trị thấp" bằng số liệu mô phỏng
🎯 Minh hoạ ví dụ kinh điển "cái cân lệch 2kg": tạo một phép đo có sai số HỆ THỐNG cố định, cho thấy độ tin cậy (alpha) vẫn có thể cao dù phép đo không chính xác.

In [5]:
rng = np.random.default_rng(2026)
true_weight = rng.normal(60, 10, 240)  # cân nặng thật
# "Cân lỗi": luôn báo nặng hơn 2kg, cộng thêm nhiễu đo nhỏ (đo 3 lần bằng cùng 1 cân lỗi)
bias = 2.0
measured_1 = true_weight + bias + rng.normal(0, 0.3, 240)
measured_2 = true_weight + bias + rng.normal(0, 0.3, 240)
measured_3 = true_weight + bias + rng.normal(0, 0.3, 240)

fake_scale = pd.DataFrame({"lan1": measured_1, "lan2": measured_2, "lan3": measured_3})
alpha_fake, _ = cronbach_alpha(fake_scale)
mean_error = (fake_scale.mean(axis=1) - true_weight).mean()
print(f"Cronbach's alpha của 'cân lỗi' (3 lần đo): {alpha_fake:.4f}  <- RẤT CAO (tin cậy tốt)")
print(f"Sai số trung bình so với cân nặng THẬT: {mean_error:+.2f} kg  <- LUÔN lệch (giá trị kém)")

Cronbach's alpha của 'cân lỗi' (3 lần đo): 0.9997  <- RẤT CAO (tin cậy tốt)
Sai số trung bình so với cân nặng THẬT: +2.00 kg  <- LUÔN lệch (giá trị kém)


#### 📤 Đầu ra thật
Alpha của "cân lỗi" rất cao (>0,95 — 3 lần đo gần như giống hệt nhau) nhưng sai số trung bình so với cân nặng THẬT luôn là +2,00kg. Minh chứng bằng số liệu mô phỏng: **độ tin cậy cao KHÔNG đảm bảo độ giá trị cao** — bài học cốt lõi của Module 12.